# E1 received-case lig_inter ordering
Owner: uibcdf/dockingmt#78. External finite study policy; existing rank_by and native records remain unchanged.
Execute only with the frozen roots, registered prescription and monitored resource limits.


In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import sys
from copy import deepcopy
from decimal import Decimal
from pathlib import Path

root = Path(os.environ['DOCKINGMT_E1_ROOT'])
output = Path(os.environ['DOCKINGMT_E1_ORDERING_OUTPUT'])
output.mkdir(parents=True, exist_ok=False)
ROOTS = {
    'construction': '45b3665ba09b6979aa1bcdf1bea0ac3e8102f081fefbfc51348c55bc71405df3',
    'scoring': 'bf214aa7e48f15a95051411ed3bd54969f78c811503050c6969d6f5aa2898e93',
}
directories = {
    'construction': root / 'devguide/validation/data/e1_complete_pose_2026-11-06',
    'scoring': root / 'devguide/validation/data/e1_native_rescoring_2026-11-06',
}
manifests = {}
for name, directory in directories.items():
    data = (directory / 'manifest.json').read_bytes()
    assert hashlib.sha256(data).hexdigest() == ROOTS[name]
    manifest = json.loads(data)
    manifests[name] = manifest
    for row in manifest['files']:
        data = (directory / row['path']).read_bytes()
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        if row['path'].endswith('.gz'):
            decoded = gzip.decompress(data)
            assert (
                len(decoded) == row['decoded_bytes']
                and hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']
            )


def document(stage, name):
    rows = [
        r
        for r in manifests[stage]['files']
        if r['path'] == name or r.get('original_path') == name
    ]
    assert len(rows) == 1, name
    row = rows[0]
    data = (directories[stage] / row['path']).read_bytes()
    if row['path'].endswith('.gz'):
        data = gzip.decompress(data)
    return json.loads(data)


def seal(value):
    return hashlib.sha256(
        json.dumps(
            value, sort_keys=True, separators=(',', ':'), allow_nan=False
        ).encode()
    ).hexdigest()


def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, sort_keys=True, indent=2, allow_nan=False)
        stream.write('\n')


roles = {
    'association': 'association.json',
    'full18': 'full18_witness.json',
    'prepared13': 'prepared13_witness.json',
    'read13': 'read13_witness.json',
    'prepared': 'prepared.json',
    'values': 'prepared_values.json',
    'full_mechanics': 'full18_mechanics_inspection.json',
    'original_charge': 'charge_assignment.json',
    'original_types': 'typing_assignment.json',
    'charge_audit': 'charge_audit.json',
    'export_map': 'export_map.json',
    'tree': 'written_tree.json',
    'lineage': 'source9_to_written_map.json',
    'hydrogens': 'hydrogen_addition.json',
}
proofs = []
for index in range(1, 6):
    proof = {
        role: document('construction', f'normal/pose_{index}/{name}')
        for role, name in roles.items()
    }
    proof['files'] = [
        r
        for r in manifests['construction']['files']
        if r['path'].startswith(f'normal/pose_{index}/')
    ]
    proofs.append(proof)
collection_record = document('scoring', 'scored_collection.json')
pristine = dict(
    roots=deepcopy(ROOTS),
    records=collection_record['poses'],
    proofs=proofs,
    contract=document('construction', 'prospective_scoring_contract.json'),
    native_receipt=document('scoring', 'receipt.json'),
    qualification=document('scoring', 'backend_qualification.json'),
)
pristine_seal = seal(pristine)

In [ ]:
from datetime import datetime, timezone

import pyunitwizard as puw
from pyunitwizard.record import QuantityRecord, RecordError
from smonitor.integrations import CatalogException

import dockingmt as dmt

budgets = dict(
    admission_attempts=70,
    admitted_studies=4,
    expected_refusals=66,
    audit_pose=24,
    quantity_decodes=100,
    rank_refusals=4,
    tie_controls=2,
    detachment_controls=2,
)
counts = dict.fromkeys(budgets, 0)


def journal(phase, **fields):
    with (output / 'admissions.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    phase=phase, at_utc=datetime.now(timezone.utc).isoformat(), **fields
                ),
                allow_nan=False,
            )
            + '\n'
        )
        stream.flush()


class StudyRefusal(Exception):
    def __init__(self, code, position=None):
        self.code = code
        self.position = position
        journal(
            'refused',
            attempt=counts['admission_attempts'],
            code=code,
            position=position,
        )
        super().__init__(code)


def require(condition, code, position=None):
    if not condition:
        raise StudyRefusal(code, position)


def consume(name):
    assert counts[name] < budgets[name], name
    counts[name] += 1


def charge_values(proof, position):
    consume('quantity_decodes')
    try:
        record = QuantityRecord.from_dict(proof['values']['partial_charge'])
        require(record.field is None, 'charge_record_invalid', position)
        quantity = record.to_quantity(field=None, unit='elementary_charge')
        values = puw.get_value(
            quantity, to_unit='elementary_charge', value_type=list, dtype=float
        )
    except StudyRefusal:
        raise
    except RecordError as error:
        raise StudyRefusal('charge_record_invalid', position) from error
    require(
        len(values) == 13 and all(math.isfinite(x) for x in values),
        'charge_values_disagree',
        position,
    )
    return values


ORDER = [6, 7, 8, 9, 10, 11, 12, 0, 1, 2, 3, 4, 5]
INVERSE = [7, 8, 9, 10, 11, 12, 0, 1, 2, 3, 4, 5, 6]
RETAINED = [0, 1, 2, 3, 4, 5, 6, 7, 8, 14, 15, 16, 17]


def receive_case_component(candidate, method='vina', component='lig_inter'):
    consume('admission_attempts')
    journal(
        'started',
        attempt=counts['admission_attempts'],
        method=method,
        component=component,
    )
    require(method in ['vina', 'vinardo'], 'method_pool_outside_profile')
    require(component == 'lig_inter', 'component_outside_profile')
    require(candidate['roots'] == ROOTS, 'custody_root_not_received')
    require(
        candidate['native_receipt']['input_sha256']['manifest']
        == ROOTS['construction'],
        'construction_scoring_link_mismatch',
    )
    records = candidate['records']
    proofs = candidate['proofs']
    require(len(records) == len(proofs) == 5, 'population_not_closed')
    require(len({r['pose_id'] for r in records}) == 5, 'duplicate_pose_id')
    qualification = candidate['qualification']
    require(
        candidate['native_receipt']['counts']['score_checked'] == 10
        and candidate['native_receipt']['engine_payload_unchanged'],
        'native_receiving_incomplete',
    )
    require(qualification['versions']['vina'] == '1.2.7', 'native_receiving_incomplete')
    selected = method + '.lig_inter'
    rows = []
    for offset, (record, proof) in enumerate(zip(records, proofs)):
        position = offset + 1
        source = pristine['records'][offset]
        received = pristine['proofs'][offset]
        require(
            record['pose_id'] == received['association']['new_pose_id'],
            'pose_identity_not_received',
            position,
        )
        require(
            record['partner_state_id'] == source['partner_state_id']
            and record['receptor_state_id'] == source['receptor_state_id'],
            'state_not_received',
            position,
        )
        require(record['rank'] is None, 'rank_not_received', position)
        definition = record['metadata']['score_definitions'][selected]
        require(
            definition['method'] == f'AutoDock Vina/{method}'
            and definition['method_version'] == '1.2.7',
            'method_not_received',
            position,
        )
        require(
            definition['component'] == 'lig_inter'
            and definition['kind'] == 'empirical'
            and definition['preferred_direction'] is None,
            'descriptor_not_received',
            position,
        )
        require(
            definition['unit'] == 'kilocalorie / mole',
            'unit_representation_outside_profile',
            position,
        )
        value = record['scores'][selected]
        require(
            type(value) in [int, float]
            and type(value) is not bool
            and math.isfinite(value),
            'invalid_reported_value',
            position,
        )
        decimal = Decimal(str(value))
        require(
            decimal == decimal.quantize(Decimal('0.001')),
            'reported_precision_outside_profile',
            position,
        )
        history = record['metadata'].get('scoring_history', [])
        require(
            len(history) == 2
            and [e['score_name'] for e in history] == ['vina', 'vinardo'],
            'evaluation_history_incomplete',
            position,
        )
        evaluation = history[0 if method == 'vina' else 1]
        reference = evaluation.get('reference_energy')
        require(
            isinstance(reference, dict)
            and set(reference)
            == {'schema_version', 'policy', 'score_name', 'value', 'unit'},
            'reference_documentary_incomplete',
            position,
        )
        require(
            reference
            == dict(
                schema_version='1.0',
                policy='native_current_conformation',
                score_name=method + '.lig_intra_best_pose',
                value=record['scores'][method + '.lig_intra_best_pose'],
                unit='kcal/mol',
            ),
            'reference_record_inconsistent',
            position,
        )
        context = definition['context']
        require(
            set(context)
            == set(source['metadata']['score_definitions'][selected]['context']),
            'context_fields_outside_profile',
            position,
        )
        require(
            context['reference_energy_policy'] == 'native_current_conformation'
            and context['stage'] == 'scoring',
            'context_not_received',
            position,
        )
        require(
            context['receptor_sha256'] == pristine['contract']['receptor_sha256'],
            'receptor_not_received',
            position,
        )
        require(
            context['partner_sha256']
            == received['association']['dockingmt_export_sha256'],
            'partner_input_not_received',
            position,
        )
        require(
            context['backend_box'] == pristine['contract']['backend_box'],
            'box_not_received',
            position,
        )
        require(
            context['weights']
            == pristine['contract']['expected_factory_weights'][method],
            'weights_not_received',
            position,
        )
        require(
            context['grid_spacing'] == pristine['contract']['grid_spacing'],
            'spacing_not_received',
            position,
        )
        require(
            context['preparation_assessment']
            == dict(receptor='unassessed', partner='unassessed'),
            'preparation_policy_not_received',
            position,
        )
        require(
            evaluation['backend_version'] == '1.2.7'
            and evaluation['dockingmt_version']
            == qualification['versions']['dockingmt']
            and evaluation['molsysmt_version'] == qualification['versions']['molsysmt'],
            'producer_not_received',
            position,
        )
        require(
            evaluation['protocol']
            == source['metadata']['scoring_history'][0 if method == 'vina' else 1][
                'protocol'
            ]
            and evaluation['applied_parameters']['cpu'] == 1
            and evaluation['applied_parameters']['seed'] == 7,
            'protocol_not_received',
            position,
        )
        try:
            dmt.verify_captured_inputs(evaluation['backend_artifacts'])
        except CatalogException as error:
            raise StudyRefusal('capture_integrity_failed', position) from error
        require(
            evaluation['backend_artifacts']
            == source['metadata']['scoring_history'][0 if method == 'vina' else 1][
                'backend_artifacts'
            ],
            'captured_input_not_received',
            position,
        )
        for role, n in [('full18', 18), ('prepared13', 13), ('read13', 13)]:
            witness = proof.get(role)
            require(isinstance(witness, dict), 'chemical_evidence_missing', position)
            identity = [[[i, i] for i in range(n)]]
            require(
                witness['schema'] == 'molsysmt.atom_correspondences@1'
                and witness['method'] == 'declared_graph_isomorphism'
                and witness['rule_version'] == 'declared_graph_isomorphism@1'
                and witness['n_atoms'] == witness['reference_n_atoms'] == n
                and witness['chemical_state_index']
                == witness['reference_chemical_state_index']
                == 0
                and witness['search_scope'] == 'supplied_atom_correspondences'
                and witness['search_complete'] is True
                and witness['n_candidate_matches'] == witness['max_matches'] == 1
                and witness['atom_correspondences']
                == witness['candidate_atom_correspondences']
                == identity
                and witness['source_graph_digest']
                == witness['reference_graph_digest']
                == pristine['proofs'][0][role]['reference_graph_digest'],
                'chemical_witness_not_received',
                position,
            )
        prepared = proof['prepared']
        md = prepared['metadata']
        q = charge_values(proof, position)
        require(
            len(prepared['charges']) == 13
            and all(
                math.isclose(a, b, rel_tol=0, abs_tol=1e-12)
                for a, b in zip(q, prepared['charges'])
            )
            and all(
                math.isclose(a, b, rel_tol=0, abs_tol=1e-12)
                for a, b in zip(q, pristine['proofs'][0]['prepared']['charges'])
            ),
            'charge_values_disagree',
            position,
        )
        require(
            prepared['atom_types']
            == proof['values']['atom_ff_type']
            == pristine['proofs'][0]['prepared']['atom_types'],
            'atom_types_disagree',
            position,
        )
        require(
            proof['charge_audit']['assessment'] == 'consistent'
            and proof['charge_audit']['charge_unit'] == 'elementary_charge'
            and abs(math.fsum(q) - 1) <= 1e-6,
            'charge_evidence_not_received',
            position,
        )
        full = proof['full_mechanics']
        ref_full = pristine['proofs'][0]['full_mechanics']
        require(
            full['native_partial_charge_unit'] == 'elementary_charge'
            and len(full['native_values']['partial_charge']) == 18
            and full['native_values']['atom_ff_type']
            == ref_full['native_values']['atom_ff_type']
            and all(
                math.isclose(a, b, rel_tol=0, abs_tol=1e-12)
                for a, b in zip(
                    full['native_values']['partial_charge'],
                    ref_full['native_values']['partial_charge'],
                )
            ),
            'full_mechanics_not_received',
            position,
        )
        require(
            proof['original_charge']['method'] == 'gasteiger_marsili'
            and proof['original_charge']['n_atoms'] == 18
            and proof['original_types']['method'] == 'chemical_environment'
            and proof['original_types']['typing_scheme'] == 'autodock4'
            and proof['original_types']['n_atoms'] == 18,
            'named_assignment_not_received',
            position,
        )
        require(
            md['projected_partial_charge_assignment']['status'] == 'derived'
            and md['projected_partial_charge_assignment'].get('atom_source_indices')
            == RETAINED
            and md['projected_atom_type_assignment']['status'] == 'derived'
            and md['projected_atom_type_assignment']['n_atoms'] == 18
            and md['projected_atom_type_assignment'].get('atom_source_indices')
            == RETAINED,
            'mechanical_projection_not_received',
            position,
        )
        require(
            prepared['n_atoms'] == 13
            and prepared['torsion_dof'] == 1
            and prepared['pdbqt_atom_indices'] == ORDER
            and md['active_torsion_bonds'] == [[0, 6]],
            'torsion_policy_not_received',
            position,
        )
        tree = proof['tree']
        mapping = proof['export_map']
        require(
            tree['branch_atom_pairs'] == [[0, 7]]
            and tree['branch_fragment_pairs'] == [[0, 1]]
            and tree['fragment_offsets'] == [0, 7, 13]
            and tree['fragment_atom_indices'] == list(range(13))
            and tree['torsdof'] == 1
            and tree['atom_ids'] == mapping['output_atom_ids'],
            'written_tree_not_received',
            position,
        )
        require(
            mapping['output_to_source_atom_indices'] == ORDER
            and mapping['source_to_output_atom_indices'] == INVERSE,
            'export_axis_not_received',
            position,
        )
        require(
            mapping['payload_sha256']
            == received['association']['native_export_sha256'],
            'native_export_binding_mismatch',
            position,
        )
        require(
            proof['association']['dockingmt_export_sha256']
            == context['partner_sha256'],
            'submitted_export_binding_mismatch',
            position,
        )
        require(
            mapping['partial_charge_before_rounding']
            == [prepared['charges'][i] for i in ORDER]
            and mapping['output_atom_ff_types']
            == [prepared['atom_types'][i] for i in ORDER],
            'written_parameters_not_received',
            position,
        )
        require(
            proof['lineage']['n_source_atoms'] == 9
            and proof['lineage']['n_output_atoms'] == 13
            and proof['lineage']['output_to_source_atom_indices']
            == [6, 7, 8, -1, -1, -1, -1, 0, 1, 2, 3, 4, 5],
            'generated_atom_lineage_not_received',
            position,
        )
        require(
            proof['hydrogens']['n_added_hydrogens'] == 9
            and not proof['hydrogens']['dropped_attributes'],
            'hydrogen_recipe_not_received',
            position,
        )
        consume('audit_pose')
        audit = dmt.audit_pose(record)
        require(audit['status'] == 'consistent', 'source_record_inconsistent', position)
        require(seal(record) == seal(source), 'candidate_record_not_received', position)
        require(
            seal(proof) == seal(received), 'candidate_evidence_not_received', position
        )
        rows.append(
            dict(
                source_position=position,
                pose_id=record['pose_id'],
                reported_value=dict(value=value, unit=definition['unit']),
                descriptor=deepcopy(definition),
                reference_record=deepcopy(reference),
                source_record_snapshot_sha256=seal(source),
                received_evidence_snapshot_sha256=seal(received),
                received_files=deepcopy(proof['files']),
                chemical_witnesses={
                    role: deepcopy(proof[role])
                    for role in ['full18', 'prepared13', 'read13']
                },
                gates=dict(
                    custody='received',
                    chemical='received_reports_only',
                    mechanical='received_values_assignments',
                    tree_axis='received',
                    fixed_model='received_case',
                    record_audit='consistent',
                ),
                chemical_validity='unassessed',
                source_build_correspondence='unassessed',
            )
        )
    require(
        seal(candidate['contract']) == seal(pristine['contract'])
        and seal(candidate['qualification']) == seal(pristine['qualification'])
        and seal(candidate['native_receipt']) == seal(pristine['native_receipt']),
        'case_receiving_not_received',
    )
    consume('admitted_studies')
    journal('admitted', attempt=counts['admission_attempts'], method=method)
    return dict(
        schema='dockingmt.e1_lig_inter_ordering_case@1',
        owner='uibcdf/dockingmt#78',
        policy='received_case_reported_lig_inter_lower_first@1',
        roots=deepcopy(ROOTS),
        method=method,
        selected_score=selected,
        direction='lower_reported_value_first',
        original_preferred_direction=None,
        reported_resolution=dict(value=0.001, unit='kcal/mol'),
        allowed_context_variation=[
            'partner_sha256 bound individually to its received complete pose'
        ],
        source_rows=rows,
        scope='Finite conditional empirical component ordering; no affinity, pooled methods, historical-energy comparison, shared unbound reference or modification of native records.',
    )


def order_rows(rows):
    ordered = sorted(rows, key=lambda row: Decimal(str(row['reported_value']['value'])))
    groups = []
    for row in ordered:
        key = str(Decimal(str(row['reported_value']['value'])))
        if not groups or Decimal(groups[-1]['reported_decimal']) != Decimal(key):
            groups.append(
                dict(
                    reported_decimal=key, unit=row['reported_value']['unit'], members=[]
                )
            )
        groups[-1]['members'].append(
            dict(source_position=row['source_position'], pose_id=row['pose_id'])
        )
    return dict(
        groups=groups,
        display_order=[row['pose_id'] for row in ordered],
        ties='Equal published value; source order within a tie is presentation only',
    )

In [ ]:
# Documentary mutations remain separate from the immutable empirical population.
def change_context(candidate, key, value):
    candidate['records'][0]['metadata']['score_definitions']['vina.lig_inter'][
        'context'
    ][key] = value


def mutate(candidate, label):
    record = candidate['records'][0]
    proof = candidate['proofs'][0]
    if label == 'altered_capture':
        artifact = record['metadata']['scoring_history'][0]['backend_artifacts'][
            'partner'
        ]
        artifact['content_base64'] = 'A' + artifact['content_base64'][1:]
    elif label == 'unreceived_partner_input':
        change_context(candidate, 'partner_sha256', '0' * 64)
    elif label == 'missing_graph_witness':
        proof['prepared13'] = None
    elif label == 'changed_graph_witness':
        proof['prepared13']['source_graph_digest'] = '0' * 64
    elif label == 'altered_charge':
        proof['prepared']['charges'][0] += 0.01
    elif label == 'altered_type':
        proof['prepared']['atom_types'][0] = 'C'
    elif label == 'altered_tree':
        proof['tree']['branch_atom_pairs'] = [[1, 7]]
    elif label == 'altered_export_axis':
        proof['export_map']['output_to_source_atom_indices'][0] = 0
    elif label == 'changed_receptor':
        change_context(candidate, 'receptor_sha256', '0' * 64)
    elif label == 'changed_partner_state':
        record['partner_state_id'] = 'other-declaration'
    elif label == 'changed_receptor_state':
        record['receptor_state_id'] = 'other-declaration'
    elif label == 'changed_weights':
        change_context(candidate, 'weights', [0.0] * 7)
    elif label == 'changed_box':
        change_context(
            candidate,
            'backend_box',
            dict(center=[0.0, 0.0, 0.0], size=[24.0, 24.0, 24.0], unit='angstrom'),
        )
    elif label == 'changed_spacing':
        change_context(candidate, 'grid_spacing', dict(value=0.4, unit='angstrom'))
    elif label == 'wrong_unit_dimension':
        record['metadata']['score_definitions']['vina.lig_inter']['unit'] = 'nm'
    elif label == 'outside_energy_representation':
        record['metadata']['score_definitions']['vina.lig_inter']['unit'] = 'kJ/mol'
    elif label == 'duplicate_pose_id':
        candidate['records'][1]['pose_id'] = record['pose_id']
    elif label == 'nonfinite_score':
        record['scores']['vina.lig_inter'] = float('nan')
    elif label == 'boolean_score':
        record['scores']['vina.lig_inter'] = True
    elif label == 'excess_precision':
        record['scores']['vina.lig_inter'] = float(
            Decimal(str(record['scores']['vina.lig_inter'])) + Decimal('0.00001')
        )
    elif label == 'missing_reference':
        del record['metadata']['scoring_history'][0]['reference_energy']
    elif label == 'contradictory_reference':
        record['metadata']['scoring_history'][0]['reference_energy']['value'] += 1
    elif label == 'unknown_context_field':
        change_context(candidate, 'unreceived_setting', 'unassessed')
    elif label == 'changed_backend_version':
        record['metadata']['score_definitions']['vina.lig_inter']['method_version'] = (
            'unknown'
        )
    elif label == 'changed_type_method':
        proof['original_types']['method'] = 'unreceived-model'
    elif label == 'missing_assignment_projection':
        del proof['prepared']['metadata']['projected_atom_type_assignment'][
            'atom_source_indices'
        ]
    elif label == 'wrong_export_hash_axis':
        proof['export_map']['payload_sha256'] = proof['association'][
            'dockingmt_export_sha256'
        ]
    elif label == 'coherently_changed_score':
        value = float(
            Decimal(str(record['scores']['vina.lig_inter'])) + Decimal('0.001')
        )
        record['scores']['vina.lig_inter'] = value
        record['metadata']['scoring_history'][0]['scores']['vina.lig_inter'] = value
    elif label == 'changed_custody_root':
        candidate['roots']['scoring'] = '0' * 64
    elif label == 'changed_receiving_link':
        candidate['native_receipt']['input_sha256']['manifest'] = '0' * 64
    elif label == 'incomplete_population':
        candidate['records'].pop()
    else:
        raise AssertionError('Unregistered mutation ' + label)


CONTROLS = [
    ('altered_capture', 'capture_integrity_failed'),
    ('unreceived_partner_input', 'partner_input_not_received'),
    ('missing_graph_witness', 'chemical_evidence_missing'),
    ('changed_graph_witness', 'chemical_witness_not_received'),
    ('altered_charge', 'charge_values_disagree'),
    ('altered_type', 'atom_types_disagree'),
    ('altered_tree', 'written_tree_not_received'),
    ('altered_export_axis', 'export_axis_not_received'),
    ('changed_receptor', 'receptor_not_received'),
    ('changed_partner_state', 'state_not_received'),
    ('changed_receptor_state', 'state_not_received'),
    ('changed_weights', 'weights_not_received'),
    ('changed_box', 'box_not_received'),
    ('changed_spacing', 'spacing_not_received'),
    ('wrong_unit_dimension', 'unit_representation_outside_profile'),
    ('outside_energy_representation', 'unit_representation_outside_profile'),
    ('duplicate_pose_id', 'duplicate_pose_id'),
    ('nonfinite_score', 'invalid_reported_value'),
    ('boolean_score', 'invalid_reported_value'),
    ('excess_precision', 'reported_precision_outside_profile'),
    ('missing_reference', 'reference_documentary_incomplete'),
    ('contradictory_reference', 'reference_record_inconsistent'),
    ('total_component', 'component_outside_profile'),
    ('mixed_methods', 'method_pool_outside_profile'),
    ('unknown_context_field', 'context_fields_outside_profile'),
    ('changed_backend_version', 'method_not_received'),
    ('changed_type_method', 'named_assignment_not_received'),
    ('missing_assignment_projection', 'mechanical_projection_not_received'),
    ('wrong_export_hash_axis', 'native_export_binding_mismatch'),
    ('coherently_changed_score', 'candidate_record_not_received'),
    ('changed_custody_root', 'custody_root_not_received'),
    ('changed_receiving_link', 'construction_scoring_link_mismatch'),
    ('incomplete_population', 'population_not_closed'),
]
assert len(CONTROLS) == 33
caller_policy = puw.configure.report()
profiles = [
    (
        'normal',
        [
            'nm',
            'ps',
            'kelvin',
            'mole',
            'dalton',
            'e',
            'kJ/mol',
            'kJ/(mol*nm)',
            'kJ/(mol*nm**2)',
            'radians',
        ],
    ),
    (
        'pm_fs',
        [
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ],
    ),
]
observations = []
for profile, units in profiles:
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=units):
        active_policy = puw.configure.report()
        admitted = []
        reader_inputs = []
        for method in ['vina', 'vinardo']:
            reader_input = deepcopy(pristine)
            before_input = seal(reader_input)
            report = receive_case_component(reader_input, method)
            assert seal(reader_input) == before_input
            reader_inputs.append(reader_input)
            report['ordering'] = order_rows(report['source_rows'])
            save(directory / (method + '_component_order.json'), report)
            admitted.append(report)
        refusals = []
        for label, expected in CONTROLS:
            candidate = deepcopy(pristine)
            method = 'vina'
            component = 'lig_inter'
            if label == 'total_component':
                component = 'total'
            elif label == 'mixed_methods':
                method = 'vina+vinardo'
            else:
                mutate(candidate, label)
            try:
                receive_case_component(candidate, method, component)
            except StudyRefusal as error:
                assert error.code == expected, (label, error.code, expected)
                consume('expected_refusals')
                refusals.append(
                    dict(
                        control=label,
                        code=error.code,
                        position=error.position,
                        scope='Documentary mutation of a copy; no new molecular/native observation',
                        ordered_subset_returned=False,
                    )
                )
            else:
                raise AssertionError('Control did not refuse: ' + label)
        save(directory / 'refusals.json', refusals)
        synthetic = [
            dict(
                source_position=i,
                pose_id=f'synthetic-{i}',
                reported_value=dict(value=value, unit='kcal/mol'),
            )
            for i, value in [(1, 2.0), (2, 1.0), (3, 1)]
        ]
        ties = order_rows(synthetic)
        assert (
            ties['display_order'] == ['synthetic-2', 'synthetic-3', 'synthetic-1']
            and len(ties['groups']) == 2
        )
        assert [m['source_position'] for m in ties['groups'][0]['members']] == [2, 3]
        consume('tie_controls')
        save(
            directory / 'synthetic_ties.json',
            dict(
                scope='Invented software values, not empirical observations',
                result=ties,
            ),
        )
        report_snapshot = deepcopy(admitted[0])
        detached = admitted[0]
        detached['source_rows'][0]['descriptor']['context']['receptor_sha256'] = (
            'changed-report-copy'
        )
        assert seal(pristine) == pristine_seal
        assert seal(reader_inputs[0]) == pristine_seal
        admitted[0] = report_snapshot
        consume('detachment_controls')
        collection = dmt.DockingResult.from_dict(deepcopy(collection_record))
        collection_before = collection.to_dict()
        rank_refusals = []
        for method in ['vina', 'vinardo']:
            try:
                collection.rank_by(method + '.lig_inter')
            except CatalogException as error:
                assert 'identical declared definitions and contexts' in str(error)
                consume('rank_refusals')
                rank_refusals.append(
                    dict(
                        method=method,
                        score_name=method + '.lig_inter',
                        code=getattr(error, 'code', None),
                        message=str(error),
                    )
                )
            else:
                raise AssertionError(
                    'Strict rank_by unexpectedly admitted different contexts'
                )
        assert collection.to_dict() == collection_before
        assert seal(pristine) == pristine_seal
        save(directory / 'strict_rank_by_refusals.json', rank_refusals)
        assert puw.configure.report() == active_policy
        observations.append(
            dict(
                profile=profile,
                admitted_methods=2,
                refused_controls=len(refusals),
                strict_rank_refusals=2,
                source_unchanged=seal(pristine) == pristine_seal,
                display_orders={
                    row['method']: row['ordering']['display_order'] for row in admitted
                },
            )
        )
    assert puw.configure.report() == caller_policy

In [ ]:
assert len(observations) == 2
for method in ['vina', 'vinardo']:
    assert (output / 'normal' / (method + '_component_order.json')).read_bytes() == (
        output / 'pm_fs' / (method + '_component_order.json')
    ).read_bytes()
assert (
    counts['admission_attempts'] == 70
    and counts['admitted_studies'] == 4
    and counts['expected_refusals'] == 66
)
assert (
    counts['rank_refusals'] == 4
    and counts['tie_controls'] == counts['detachment_controls'] == 2
)
assert seal(pristine) == pristine_seal
for stage, directory in directories.items():
    assert (
        hashlib.sha256((directory / 'manifest.json').read_bytes()).hexdigest()
        == ROOTS[stage]
    )
    for row in manifests[stage]['files']:
        data = (directory / row['path']).read_bytes()
        assert hashlib.sha256(data).hexdigest() == row['sha256']
assert not any(
    name == 'vina'
    or name.startswith('vina.')
    or name == 'rdkit'
    or name.startswith('rdkit.')
    or name == 'molsysviewer'
    or name.startswith('molsysviewer.')
    for name in sys.modules
)
save(
    output / 'receipt.json',
    dict(
        schema='dockingmt.e1_lig_inter_ordering_receiving@1',
        owner='uibcdf/dockingmt#78',
        roots=ROOTS,
        python=platform.python_version(),
        versions={
            name: metadata.version(name)
            for name in [
                'dockingmt',
                'molsysmt',
                'pyunitwizard',
                'argdigest',
                'depdigest',
                'smonitor',
            ]
        },
        counts=counts,
        profiles=observations,
        distinct_component_studies=2,
        reader_policy_repetitions=2,
        input_collection_unchanged=True,
        all_source_files_unchanged=True,
        new_vina_calls=0,
        new_molecular_operations=0,
        excluded_operation_basis='Inspected record-only public routes; no molecular or native counter hooks were installed.',
        new_empirical_values=0,
        scope='Finite case-specific lower-reported-lig_inter ordering, separate per method; no public API or source descriptor/rank modification, affinity or total/shared-reference comparison.',
        full_e1_accepted=False,
        installed_acceptance=False,
    ),
)
print(
    json.dumps(
        dict(
            admitted_studies=4,
            distinct_method_orders=2,
            documentary_refusals=66,
            strict_rank_refusals=4,
            new_vina_calls=0,
        )
    )
)